# ML Anomaly Detection

> **Purpose**: Demonstrate the `anomaly.py` module — detect anomalous transactions using two ML models and combine results via consensus.

**All business logic lives in `../anomaly.py`. This notebook only imports and calls those functions.**

### Models used
| Model | Type | Library |
|-------|------|---------|
| **Isolation Forest** | Tree-based (isolation) | scikit-learn |
| **Local Outlier Factor (LOF)** | Density-based (neighbourhood) | scikit-learn |
| **Consensus** | Rows flagged by **both** models | Custom logic |

Using consensus reduces false positives: a record must be flagged by **both** methods to be considered a confirmed anomaly.

> **Note**: Anomaly detection runs on the **cleaned** dataframe where `Price` is already numeric.

> **Previous step**: `statistics.ipynb` | **Next step**: `scoring.ipynb`

In [ ]:
import sys
import os
import json

sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from cleaning import load_dataset, run_cleaning
from anomaly import run_ml_anomalies

DATA_PATH = os.path.join("..", "data", "dataset_ecommerce_transactions_data.csv")
df_raw   = load_dataset(DATA_PATH)
df_clean = run_cleaning(df_raw)

print(f"Dataset loaded and cleaned: {df_clean.shape}")
print(f"Numeric features available: {df_clean.select_dtypes(include='number').columns.tolist()}")

## Run Anomaly Detection

In [ ]:
# contamination=0.05 means we expect ~5% of rows to be anomalous
anomaly_result = run_ml_anomalies(df_clean, contamination=0.05, random_state=42)

# Print summary (exclude large lists)
summary_keys = [k for k in anomaly_result if k not in ("consensus_indices", "anomaly_records")]
summary = {k: anomaly_result[k] for k in summary_keys}

print("=== Anomaly Detection Summary ===")
print(json.dumps(summary, indent=2))

## Model Comparison Table

In [ ]:
comparison = pd.DataFrame([
    {
        "Model": "Isolation Forest",
        "Anomalies":  anomaly_result.get("isolation_forest_anomalies", 0),
        "Percentage": anomaly_result.get("isolation_forest_pct", 0.0),
    },
    {
        "Model": "Local Outlier Factor",
        "Anomalies":  anomaly_result.get("lof_anomalies", 0),
        "Percentage": anomaly_result.get("lof_pct", 0.0),
    },
    {
        "Model": "Consensus (Both)",
        "Anomalies":  anomaly_result.get("consensus_anomalies", 0),
        "Percentage": anomaly_result.get("consensus_pct", 0.0),
    },
])
print(comparison.to_string(index=False))

## Sample Anomalous Records

In [ ]:
records = anomaly_result.get("anomaly_records", [])
if records:
    print(f"Total consensus anomalies: {anomaly_result['consensus_anomalies']:,}")
    print(f"Showing first 5 of {len(records)} stored samples:")
    pd.DataFrame(records[:5])
else:
    print("No consensus anomalies found.")

---
## Key Takeaways

- **Isolation Forest** works by isolating points in random trees — anomalies are easier to isolate so they have shorter path lengths
- **LOF** compares a point's local density to its neighbours — points in low-density regions relative to their neighbours are flagged
- Using **consensus** (intersection of both) gives higher precision with fewer false positives vs. using either model alone
- The `contamination` parameter controls the expected anomaly rate — adjust based on domain knowledge
- The `anomaly_penalty` in `scoring.py` uses the `consensus_pct` from this result to penalise the overall quality score
- Features used: numeric columns (Quantity, Price) plus any other parseable numeric columns